# Xử lý data

In [ ]:
import os
import shutil
import hashlib
import random
from glob import glob
from collections import Counter
from pathlib import Path

# ==========================================
# CẤU HÌNH ĐƯỜNG DẪN & THÔNG SỐ
# ==========================================
thumuc_goc = r'D:\TGMT\TGMTNC\giaothongvn' 
thumuc_sach = r'D:\TGMT\TGMTNC\giaothong_final'

# CHUẨN THEO DỮ LIỆU GỐC: 0 là o_to, 1 là xe_may
class_names = {0: 'o_to', 1: 'xe_may', 2: 'xe_tai', 3: 'xe_bus', 4: 'xe_ba_gac'}

# TỰ ĐỘNG XÓA THƯ MỤC FINAL CŨ (Nếu có) ĐỂ LÀM LẠI TỪ ĐẦU
if os.path.exists(thumuc_sach):
    print(f"[*] Đang dọn dẹp thư mục cũ: {thumuc_sach}...")
    shutil.rmtree(thumuc_sach)

# ==========================================
# BƯỚC 1: QUÉT DỮ LIỆU GỐC
# ==========================================
all_images = glob(os.path.join(thumuc_goc, '**', 'images', '*.jpg'), recursive=True) + \
             glob(os.path.join(thumuc_goc, '**', 'images', '*.png'), recursive=True)
all_labels = glob(os.path.join(thumuc_goc, '**', 'labels', '*.txt'), recursive=True)

label_dict = {Path(p).stem: p for p in all_labels}
print(f"[*] Đã tìm thấy {len(all_images)} ảnh và {len(all_labels)} file nhãn.")

# ==========================================
# BƯỚC 2: LÀM SẠCH & ĐỌC CLASS
# ==========================================
print("[*] Đang làm sạch dữ liệu (Check MD5, Nhãn rỗng)...")
seen_hashes = set()
valid_data = [] 

loi_mat_nhan = loi_nhan_rong = loi_trung_lap = 0

for img_path in all_images:
    stem = Path(img_path).stem
    
    if stem not in label_dict:
        loi_mat_nhan += 1
        continue
        
    lbl_path = label_dict[stem]
    
    if os.path.getsize(lbl_path) == 0:
        loi_nhan_rong += 1
        continue
        
    with open(img_path, "rb") as f:
        file_hash = hashlib.md5(f.read()).hexdigest()
    if file_hash in seen_hashes:
        loi_trung_lap += 1
        continue
    seen_hashes.add(file_hash)
    
    classes_in_img = set()
    with open(lbl_path, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if parts:
                classes_in_img.add(int(parts[0]))
                
    if not classes_in_img:
        loi_nhan_rong += 1
        continue

    valid_data.append((img_path, lbl_path, classes_in_img))

print(f"    - Xóa {loi_mat_nhan} ảnh mất nhãn.")
print(f"    - Xóa {loi_nhan_rong} file nhãn rỗng/lỗi.")
print(f"    - Xóa {loi_trung_lap} ảnh trùng lặp.")
print(f"    => TỔNG ẢNH SẠCH SẴN SÀNG CHIA: {len(valid_data)}")

# ==========================================
# BƯỚC 3: CHIA ĐA TẦNG (STRATIFIED SPLIT 7-2-1)
# ==========================================
print("[*] Đang tiến hành chia đa tầng (Stratified Split)...")
# Thứ tự ưu tiên bốc class hiếm: 4 > 3 > 2 > 0 (Ô tô) > 1 (Xe máy)
buckets = {4: [], 3: [], 2: [], 1: [], 0: []}

for img_path, lbl_path, classes in valid_data:
    if 4 in classes: buckets[4].append((img_path, lbl_path))
    elif 3 in classes: buckets[3].append((img_path, lbl_path))
    elif 2 in classes: buckets[2].append((img_path, lbl_path))
    elif 0 in classes: buckets[0].append((img_path, lbl_path)) # Ô tô (0) hiếm hơn xe máy
    else: buckets[1].append((img_path, lbl_path))              # Còn lại là xe máy (1)

train_set, val_set, test_set = [], [], []
random.seed(42)

for class_id, items in buckets.items():
    random.shuffle(items)
    total_in_bucket = len(items)
    t_end = int(total_in_bucket * 0.7)
    v_end = t_end + int(total_in_bucket * 0.2)
    
    train_set.extend(items[:t_end])
    val_set.extend(items[t_end:v_end])
    test_set.extend(items[v_end:])

# ==========================================
# BƯỚC 4: COPY FILE & XUẤT BÁO CÁO
# ==========================================
print("[*] Đang copy file sang thư mục mới và thống kê...")

def process_split(dataset, split_name):
    img_dir = os.path.join(thumuc_sach, split_name, 'images')
    lbl_dir = os.path.join(thumuc_sach, split_name, 'labels')
    os.makedirs(img_dir, exist_ok=True)
    os.makedirs(lbl_dir, exist_ok=True)
    
    class_counts = Counter()
    for img_src, lbl_src in dataset:
        shutil.copy(img_src, os.path.join(img_dir, Path(img_src).name))
        shutil.copy(lbl_src, os.path.join(lbl_dir, Path(lbl_src).name))
        
        with open(lbl_src, 'r') as f:
            for line in f:
                parts = line.strip().split()
                if parts: class_counts[int(parts[0])] += 1
    return class_counts

counts_train = process_split(train_set, 'train')
counts_val = process_split(val_set, 'valid')
counts_test = process_split(test_set, 'test')

# XUẤT YAML CHUẨN: 0 là o_to, 1 là xe_may
yaml_content = f"train: ../train/images\nval: ../valid/images\ntest: ../test/images\nnc: 5\nnames:\n  0: o_to\n  1: xe_may\n  2: xe_tai\n  3: xe_bus\n  4: xe_ba_gac\n"
with open(os.path.join(thumuc_sach, 'data.yaml'), 'w') as f:
    f.write(yaml_content)

# ==========================================
# BƯỚC 5: TỔNG KẾT RA MÀN HÌNH CHUẨN XÁC
# ==========================================
total_clean = len(valid_data)
def in_bao_cao(name, count_imgs, class_dict):
    percent = (count_imgs / total_clean) * 100
    print(f"\n--- TẬP {name.upper()} : {count_imgs} ẢNH ({percent:.2f}%) ---")
    for cid in range(5):
        print(f"    + {class_names[cid]}: {class_dict.get(cid, 0)}")

print("\n" + "="*40)
print(" HOÀN TẤT XỬ LÝ DỮ LIỆU ĐA TẦNG")
print("="*40)
in_bao_cao("Train", len(train_set), counts_train)
in_bao_cao("Valid", len(val_set), counts_val)
in_bao_cao("Test", len(test_set), counts_test)
print("\n[*] Data sạch và file data.yaml CHUẨN đã được xuất tại:", thumuc_sach)

In [ ]:
import os
import glob
import matplotlib.pyplot as plt

# ==========================================
# THỐNG KÊ SỐ LƯỢNG BOUNDING BOX TRONG DATASET
# ==========================================

# Thư mục chứa nhãn của bạn (Sơn kiểm tra lại đường dẫn nếu cần)
label_dirs = [
    r'D:\TGMT\TGMTNC\giaothong_final\train\labels',
    r'D:\TGMT\TGMTNC\giaothong_final\val\labels'
]

class_names = ['O To', 'Xe May', 'Xe Tai', 'Xe Bus', 'Xe Ba Gac']
class_counts = {i: 0 for i in range(len(class_names))}

total_boxes = 0

# Quét qua tất cả file .txt để đếm
for label_dir in label_dirs:
    if os.path.exists(label_dir):
        txt_files = glob.glob(os.path.join(label_dir, '*.txt'))
        for txt_file in txt_files:
            with open(txt_file, 'r') as f:
                lines = f.readlines()
                for line in lines:
                    parts = line.strip().split()
                    if len(parts) > 0:
                        class_id = int(parts[0])
                        if class_id in class_counts:
                            class_counts[class_id] += 1
                            total_boxes += 1

# --- 1. IN BÁO CÁO DẠNG TEXT ---
print("\n" + "="*50)
print(" 📊 BÁO CÁO THỐNG KÊ SỐ LƯỢNG BOUNDING BOX")
print("="*50)
for class_id, count in class_counts.items():
    print(f" - {class_names[class_id]:<15}: {count:>8} boxes")
print("-" * 50)
print(f" TỔNG CỘNG      : {total_boxes:>8} boxes")
print("="*50)

# --- 2. VẼ BIỂU ĐỒ CỘT CHO BÁO CÁO WORD ---
plt.figure(figsize=(10, 6))
# Dùng các màu sắc phân biệt rõ ràng
colors = ['#3498db', '#e74c3c', '#2ecc71', '#f1c40f', '#9b59b6']
bars = plt.bar(class_names, [class_counts[i] for i in range(len(class_names))], color=colors)

# In con số cụ thể lên đỉnh mỗi cột
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, yval + (max(class_counts.values()) * 0.02), 
             f'{int(yval):,}', ha='center', va='bottom', fontsize=12, fontweight='bold')

plt.title('THỐNG KÊ MẤT CÂN BẰNG DỮ LIỆU (DATA IMBALANCE)', fontsize=14, fontweight='bold', pad=20)
plt.xlabel('Loại phương tiện (Classes)', fontsize=12, fontweight='bold')
plt.ylabel('Tổng số lượng Khung nhận diện (Bounding Boxes)', fontsize=12, fontweight='bold')
plt.grid(axis='y', linestyle='--', alpha=0.6)

# Làm gọn giao diện
plt.tight_layout()
plt.show()

# yolov8n(baseline)

In [ ]:
import os
import glob
from ultralytics import YOLO

# ==========================================
# PHẦN 1: HUẤN LUYỆN MÔ HÌNH CƠ SỞ (BASELINE - 100 EPOCHS)
# ==========================================
print("⏳ BẮT ĐẦU HUẤN LUYỆN MÔ HÌNH BASELINE (Chạy mặc định 100 Epochs)...")

data_yaml = r'D:\TGMT\TGMTNC\giaothong_final\data.yaml'
base_dir = r'D:\TGMT\TGMTNC\Ket_Qua_Train'

# Tải mô hình gốc làm nền tảng
model = YOLO('yolov8n.pt') 

# Khởi chạy huấn luyện Baseline (100 vòng, thả rông các thông số khác)
results = model.train(
    data=data_yaml, 
    epochs=100,            # Bằng đúng 100 vòng như bản Tối ưu để công bằng
    imgsz=640,             
    device=0,              
    workers=0,             
    project=base_dir, 
    name='MoHinh_Baseline',
    exist_ok=True          
)

print("\n✅ HUẤN LUYỆN BASELINE HOÀN TẤT! HỆ THỐNG CHUYỂN SANG ĐÁNH GIÁ...")

# ==========================================
# PHẦN 2: LẤY 4 CHỈ SỐ TRỌNG SỐ (VALIDATION)
# ==========================================
list_of_files = glob.glob(os.path.join(base_dir, 'MoHinh_Baseline', 'weights', 'best.pt'))

if not list_of_files:
    print("\n❌ LỖI: Không tìm thấy file best.pt. Có thể quá trình train bị lỗi.")
else:
    best_model_path = max(list_of_files, key=os.path.getctime)
    print(f"\n[+] Đang nạp mô hình Baseline vừa chạy: {best_model_path}")

    # Chấm điểm trên tập Test
    eval_model = YOLO(best_model_path)
    metrics = eval_model.val(data=data_yaml, split='test', verbose=False)

    # In bảng kết quả
    print("\n" + "="*60)
    print(" BẢNG KẾT QUẢ ĐÁNH GIÁ MÔ HÌNH CƠ SỞ (BASELINE - 100 EPOCHS)")
    print("="*60)
    print(f" 1. Precision (Độ chính xác)   : {metrics.box.mp * 100:>6.2f} %")
    print(f" 2. Recall (Độ bao phủ)       : {metrics.box.mr * 100:>6.2f} %")
    print(f" 3. mAP@50 (Điểm tổng hợp)    : {metrics.box.map50 * 100:>6.2f} %")
    print(f" 4. mAP@50-95 (Điểm khắt khe) : {metrics.box.map * 100:>6.2f} %")
    print("="*60)

# yolov8n(đã tối ưu)

In [ ]:
import os
import glob
from ultralytics import YOLO

# ==========================================
# PHẦN 1: HUẤN LUYỆN MÔ HÌNH (TRAIN)
# ==========================================
print("🚀 BẮT ĐẦU HUẤN LUYỆN MÔ HÌNH (Max 100 Epochs)...")

data_yaml = r'D:\TGMT\TGMTNC\giaothong_final\data.yaml'
base_dir = r'D:\TGMT\TGMTNC\Ket_Qua_Train'

# Tải mô hình gốc làm nền tảng để học chuyển giao
model = YOLO('yolov8n.pt') 

# Khởi chạy huấn luyện
results = model.train(
    data=data_yaml, 
    epochs=100,            # Tối đa 100 vòng
    patience=30,           # Chống học vẹt: Dừng nếu 30 vòng không tiến bộ
    imgsz=640,             
    batch=16,              # Tối ưu cho RTX 4060 8GB
    device=0,              
    workers=0,             # Chống crash trên Windows
    fliplr=0.5,            
    flipud=0.0,            # Tuyệt đối không lật dọc
    project=base_dir, 
    name='MoHinh_100_Epoch',                     
    exist_ok=True                            
)

print("\n✅ HUẤN LUYỆN HOÀN TẤT! HỆ THỐNG CHUYỂN SANG ĐÁNH GIÁ MÔ HÌNH...")

# ==========================================
# PHẦN 2: LẤY 4 CHỈ SỐ TRỌNG SỐ (VALIDATION)
# ==========================================
# Tự động quét tìm file best.pt vừa được đẻ ra sau khi train xong
list_of_files = glob.glob(os.path.join(base_dir, '**', 'weights', 'best.pt'), recursive=True)

if not list_of_files:
    print("\n❌ LỖI: Không tìm thấy file best.pt. Có thể quá trình train bị lỗi giữa chừng do thiếu RAM.")
else:
    # Lấy chính xác cái file best.pt mới nhất
    best_model_path = max(list_of_files, key=os.path.getctime)
    print(f"\n[+] Đang nạp mô hình vừa huấn luyện: {best_model_path}")

    # Chấm điểm trên tập Test
    eval_model = YOLO(best_model_path)
    metrics = eval_model.val(data=data_yaml, split='test', verbose=False)

    # Lấy 4 chỉ số cốt lõi
    precision = metrics.box.mp    # Mean Precision
    recall = metrics.box.mr       # Mean Recall
    map50 = metrics.box.map50     # mAP@50
    map95 = metrics.box.map       # mAP@50-95

    # In bảng kết quả
    print("\n" + "="*60)
    print(" BẢNG KẾT QUẢ ĐÁNH GIÁ MÔ HÌNH ĐỀ XUẤT (best.pt)")
    print("="*60)
    print(f" 1. Precision (Độ chính xác)   : {precision * 100:>6.2f} %")
    print(f" 2. Recall (Độ bao phủ)       : {recall * 100:>6.2f} %")
    print(f" 3. mAP@50 (Điểm tổng hợp)    : {map50 * 100:>6.2f} %")
    print(f" 4. mAP@50-95 (Điểm khắt khe) : {map95 * 100:>6.2f} %")
    print("="*60)
    
    if map50 > 0.8:
        print("\n🎯 ĐÁNH GIÁ: Mô hình của bạn học rất xuất sắc (mAP > 80%)!")
        print("Sẵn sàng mang file best.pt này đi chạy Test Video đếm lưu lượng.")

# So sánh hiệu suất mô hình

In [ ]:
import os
from ultralytics import YOLO

print("\n" + "="*105)
print(" 🚀 BẮT ĐẦU QUÁ TRÌNH BENCHMARK: SO SÁNH TIẾN HÓA 3 MÔ HÌNH")
print("="*105)

# Đường dẫn dữ liệu và thư mục lưu
data_yaml = r'D:\TGMT\TGMTNC\giaothong_final\data.yaml'
save_dir = r'D:\TGMT\TGMTNC\Ket_Qua_Train'

# Khai báo đường dẫn 3 bộ não
model_goc_path = 'yolov8n.pt'
model_base_path = r'D:\TGMT\TGMTNC\Ket_Qua_Train\MoHinh_Baseline\weights\best.pt'
model_prop_path = r'D:\TGMT\TGMTNC\Ket_Qua_Train\MoHinh_100_Epoch\weights\best.pt'

# --- 1. Chấm điểm YOLOv8 Gốc (Chưa huấn luyện) ---
print("[1/3] Đang chấm điểm mô hình YOLOv8 Gốc (Tải từ mạng)...")
model_goc = YOLO(model_goc_path)
metrics_goc = model_goc.val(data=data_yaml, split='test', verbose=False, project=save_dir, name='Danh_Gia_YoloGoc', exist_ok=True)
map50_goc = metrics_goc.box.map50
speed_goc_ms = metrics_goc.speed['inference']
fps_goc = 1000 / speed_goc_ms if speed_goc_ms > 0 else 0

# --- 2. Chấm điểm Baseline (Vừa huấn luyện) ---
print("\n[2/3] Đang chấm điểm mô hình Baseline (Huấn luyện 100 Epochs mặc định)...")
model_base = YOLO(model_base_path)
metrics_base = model_base.val(data=data_yaml, split='test', verbose=False, project=save_dir, name='Danh_Gia_Baseline_HuanLuyen', exist_ok=True)
map50_base = metrics_base.box.map50
speed_base_ms = metrics_base.speed['inference']
fps_base = 1000 / speed_base_ms if speed_base_ms > 0 else 0

# --- 3. Chấm điểm Đề xuất ---
print("\n[3/3] Đang chấm điểm mô hình Đề xuất (Tối ưu siêu tham số)...")
model_prop = YOLO(model_prop_path)
metrics_prop = model_prop.val(data=data_yaml, split='test', verbose=False, project=save_dir, name='Danh_Gia_DeXuat_ToiUu', exist_ok=True)
map50_prop = metrics_prop.box.map50
speed_prop_ms = metrics_prop.speed['inference']
fps_prop = 1000 / speed_prop_ms if speed_prop_ms > 0 else 0

# --- IN BẢNG KẾT QUẢ SO SÁNH 3 MÔ HÌNH ---
print("\n\n" + "="*105)
print(f"{'TIÊU CHÍ ĐÁNH GIÁ':<30} | {'YOLOv8 GỐC (Quốc tế)':<20} | {'BASELINE (Mặc định)':<20} | {'ĐỀ XUẤT (Tối ưu)':<20}")
print("-" * 105)
print(f"{'Độ chính xác trung bình mAP50':<30} | {map50_goc*100:>18.2f} % | {map50_base*100:>18.2f} % | {map50_prop*100:>18.2f} %")
print(f"{'Độ trễ xử lý 1 ảnh (Latency)':<30} | {speed_goc_ms:>18.2f} ms | {speed_base_ms:>18.2f} ms | {speed_prop_ms:>18.2f} ms")
print(f"{'Tốc độ khung hình (FPS)':<30} | {fps_goc:>18.1f} FPS | {fps_base:>18.1f} FPS | {fps_prop:>18.1f} FPS")
print("="*105)

# Ma trận nhầm lẫn

In [ ]:
from IPython.display import Image, display

# Trỏ thẳng tới cái ảnh Ma trận nhầm lẫn chuẩn (Normalized) đã có sẵn trong thư mục của bạn
img_path = r'D:\TGMT\TGMTNC\Ket_Qua_Train\MoHinh_100_Epoch\confusion_matrix_normalized.png'

# Hiển thị thẳng ra màn hình 
print("[+] Đang lấy ảnh có sẵn từ thư mục MoHinh_100_Epoch...")
display(Image(filename=img_path, width=800))

# Loss

In [ ]:
%matplotlib inline
import pandas as pd
import matplotlib.pyplot as plt

# 1. Trỏ đến file dữ liệu thô (results.csv) của YOLO
csv_path = r'D:\TGMT\TGMTNC\Ket_Qua_Train\MoHinh_100_Epoch\results.csv'

# Đọc file và dọn dẹp khoảng trắng thừa trong tên cột
df = pd.read_csv(csv_path)
df.columns = df.columns.str.strip()

# 2. Tạo khung vẽ biểu đồ: 1 hàng, 2 cột 
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# --- BIỂU ĐỒ 1: BOX LOSS (Sai số khung viền) ---
axes[0].plot(df['epoch'], df['train/box_loss'], label='Huấn luyện (Train Box Loss)', color='#1f77b4', linewidth=2.5)
axes[0].plot(df['epoch'], df['val/box_loss'], label='Đánh giá (Val Box Loss)', color='#ff7f0e', linewidth=2.5, linestyle='--')
axes[0].set_title('ĐỘ LỖI KHUNG NHẬN DIỆN (BOX LOSS)', fontsize=14, fontweight='bold', pad=15)
axes[0].set_xlabel('Vòng lặp (Epoch)', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Giá trị Loss', fontsize=12, fontweight='bold')
axes[0].legend(fontsize=11)
axes[0].grid(True, linestyle=':', alpha=0.7)

# --- BIỂU ĐỒ 2: CLS LOSS (Sai số phân loại) ---
axes[1].plot(df['epoch'], df['train/cls_loss'], label='Huấn luyện (Train Cls Loss)', color='#2ca02c', linewidth=2.5)
axes[1].plot(df['epoch'], df['val/cls_loss'], label='Đánh giá (Val Cls Loss)', color='#d62728', linewidth=2.5, linestyle='--')
axes[1].set_title('ĐỘ LỖI PHÂN LOẠI (CLASS LOSS)', fontsize=14, fontweight='bold', pad=15)
axes[1].set_xlabel('Vòng lặp (Epoch)', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Giá trị Loss', fontsize=12, fontweight='bold')
axes[1].legend(fontsize=11)
axes[1].grid(True, linestyle=':', alpha=0.7)

# 3. Canh lề
plt.tight_layout()

# 4. LƯU THẲNG RA FILE ẢNH SIÊU NÉT (Chống lỗi không hiện hình)
save_path = r'D:\TGMT\TGMTNC\Ket_Qua_Train\MoHinh_100_Epoch\BieuDo_Loss_SieuNet.png'
plt.savefig(save_path, dpi=300, bbox_inches='tight')
print(f"[+] Đã vẽ xong! Ảnh siêu nét được lưu sẵn tại:\n{save_path}")

# 5. Ép hiển thị ra màn hình
plt.show()

# Accuracy

In [ ]:
%matplotlib inline
import pandas as pd
import matplotlib.pyplot as plt

# 1. Trỏ đến file results.csv của bạn
csv_path = r'D:\TGMT\TGMTNC\Ket_Qua_Train\MoHinh_100_Epoch\results.csv'

# Đọc file và làm sạch tên cột
df = pd.read_csv(csv_path)
df.columns = df.columns.str.strip()

# 2. Khởi tạo khung vẽ
plt.figure(figsize=(10, 6))

# 3. Vẽ 3 đường biểu diễn "Độ chính xác" (Metrics)
plt.plot(df['epoch'], df['metrics/precision(B)'], label='Precision (Độ chính xác)', color='#1f77b4', linewidth=2.5)
plt.plot(df['epoch'], df['metrics/recall(B)'], label='Recall (Độ bao phủ)', color='#ff7f0e', linewidth=2.5)
# Cho đường mAP@50 màu xanh lá, nét dày hơn vì nó là quan trọng nhất
plt.plot(df['epoch'], df['metrics/mAP50(B)'], label='mAP@50 (Điểm tổng hợp)', color='#2ca02c', linewidth=3.5)

# 4. Trang trí biểu đồ cho chuyên nghiệp
plt.title('BIỂU ĐỒ ĐÁNH GIÁ ĐỘ CHÍNH XÁC CỦA MÔ HÌNH (METRICS)', fontsize=15, fontweight='bold', pad=15)
plt.xlabel('Vòng lặp (Epoch)', fontsize=12, fontweight='bold')
plt.ylabel('Giá trị đánh giá (Từ 0.0 đến 1.0)', fontsize=12, fontweight='bold')

# Giới hạn trục Y từ 0 đến 1.1 để đồ thị có khoảng thở phía trên
plt.ylim(0, 1.1)

# Chú thích ở góc dưới bên phải
plt.legend(loc='lower right', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()

# 5. Tự động lưu file ảnh siêu nét
save_path = r'D:\TGMT\TGMTNC\Ket_Qua_Train\MoHinh_100_Epoch\BieuDo_Metrics_SieuNet.png'
plt.savefig(save_path, dpi=300)
print(f"[+] Đã vẽ xong! Ảnh siêu nét được lưu sẵn tại:\n{save_path}")

# Hiển thị trực tiếp lên Jupyter
plt.show()

# Phân phối boundingbox

In [ ]:
%matplotlib inline
import os
import glob
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Trỏ vào thư mục chứa nhãn (labels) của tập Train
label_dir = r'D:\TGMT\TGMTNC\giaothong_final\train\labels'
label_files = glob.glob(os.path.join(label_dir, '*.txt'))

print(f"[*] Đang quét {len(label_files)} file nhãn để phân tích Bounding Box...")

# 2. Thu thập dữ liệu tọa độ (x, y, w, h)
data = []
for file in label_files:
    with open(file, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) == 5:
                cls_id, x, y, w, h = map(float, parts)
                data.append([x, y, w, h])

# Đưa vào bảng Pandas để dễ vẽ
df_boxes = pd.DataFrame(data, columns=['x', 'y', 'w', 'h'])

# 3. THIẾT LẬP KHUNG VẼ BIỂU ĐỒ (1 Hàng, 2 Cột)
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
sns.set_style("whitegrid")

# --- BIỂU ĐỒ 1: BẢN ĐỒ NHIỆT VỊ TRÍ TÂM (X, Y) ---
# Dùng màu YlOrRd (Vàng - Cam - Đỏ) để thể hiện khu vực xe cộ tập trung đông nhất
sns.histplot(x=df_boxes['x'], y=df_boxes['y'], bins=50, cmap="YlOrRd", cbar=True, ax=axes[0])
axes[0].set_title('PHÂN BỐ VỊ TRÍ TÂM XE TRÊN CAMERA (X, Y)', fontsize=14, fontweight='bold', pad=15)
axes[0].set_xlabel('Tọa độ trục ngang X (Chuẩn hóa 0-1)', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Tọa độ trục dọc Y (Chuẩn hóa 0-1)', fontsize=12, fontweight='bold')
# Trục Y của ảnh tính từ trên xuống dưới nên ta phải đảo ngược trục Y
axes[0].invert_yaxis() 

# --- BIỂU ĐỒ 2: BẢN ĐỒ NHIỆT KÍCH THƯỚC (WIDTH, HEIGHT) ---
# Dùng màu Blues (Trắng - Xanh) để thể hiện kích thước phổ biến của xe
sns.histplot(x=df_boxes['w'], y=df_boxes['h'], bins=50, cmap="Blues", cbar=True, ax=axes[1])
axes[1].set_title('PHÂN BỐ KÍCH THƯỚC XE (WIDTH, HEIGHT)', fontsize=14, fontweight='bold', pad=15)
axes[1].set_xlabel('Chiều rộng Bounding Box (w)', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Chiều cao Bounding Box (h)', fontsize=12, fontweight='bold')

# 4. Trang trí và canh lề
plt.tight_layout()

# 5. LƯU ẢNH SIÊU NÉT ĐỂ NỘP BÁO CÁO
save_path = r'D:\TGMT\TGMTNC\Ket_Qua_Train\PhanPhoi_BoundingBox.png'
plt.savefig(save_path, dpi=300, bbox_inches='tight')
print(f"[+] Vẽ thành công! Ảnh siêu nét được lưu tại:\n{save_path}")

# Hiện hình ra màn hình
plt.show()

# Test thử

In [ ]:
import cv2
import numpy as np
from ultralytics import YOLO
import os

# ==========================================
# 1. CẤU HÌNH ĐƯỜNG DẪN HỆ THỐNG
# ==========================================
print("[*] Đang khởi động Hệ thống AI Giao thông...")

# Trỏ đến file trọng số tốt nhất của bạn
model_path = r'D:\TGMT\TGMTNC\Ket_Qua_Train\MoHinh_100_Epoch\weights\best.pt'
model = YOLO(model_path)

# ĐƯỜNG DẪN VIDEO CỦA BẠN ĐÃ ĐƯỢC CẬP NHẬT CHÍNH XÁC
video_input = r'D:\TGMT\TGMTNC\VIDTEST\Recording 2026-06-03 112313.mp4'
video_output = r'D:\TGMT\TGMTNC\KQTEST\KetQua_20260603103101.mp4'

# Tự động tạo thư mục KQTEST nếu trên máy bạn chưa có
os.makedirs(r'D:\TGMT\TGMTNC\KQTEST', exist_ok=True)

# Tên các lớp phương tiện
class_names = ['O To', 'Xe May', 'Xe Tai', 'Xe Bus', 'Xe Ba Gac']

# ==========================================
# 2. KHỞI TẠO VIDEO & VÙNG ĐẾM (ROI)
# ==========================================
cap = cv2.VideoCapture(video_input)

# Kiểm tra xem có đọc được video không
if not cap.isOpened():
    print(f"❌ LỖI: Không thể mở được video tại {video_input}. Bạn kiểm tra lại tên file nhé!")
else:
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(video_output, fourcc, fps, (width, height))

    # ---------------------------------------------------------
    # ⚠️ TỌA ĐỘ VÙNG ĐẾM (ROI) 
    # Nếu khung màu đỏ bị lệch lề đường, bạn chỉnh 4 số này nhé!
    # ---------------------------------------------------------
    roi_points = np.array([
        [300, 400],   # Góc trên bên trái
        [900, 400],   # Góc trên bên phải
        [1100, 650],  # Góc dưới bên phải
        [100, 650]    # Góc dưới bên trái
    ], np.int32)

    counted_ids = set()
    total_vehicles = 0
    vehicle_counts = {0: 0, 1: 0, 2: 0, 3: 0, 4: 0} 

    print(f"[+] Bắt đầu xử lý video: 20260603103101.mp4...")

    # ==========================================
    # 3. VÒNG LẶP XỬ LÝ CHÍNH
    # ==========================================
    while cap.isOpened():
        success, frame = cap.read()
        if not success:
            break # Hết video

        roi_color_flash = False 

        # Chạy YOLOv8 Tracking với BoT-SORT
        results = model.track(frame, persist=True, tracker="botsort.yaml", verbose=False)
        
        if results[0].boxes is not None and results[0].boxes.id is not None:
            boxes = results[0].boxes.xyxy.cpu().numpy()
            track_ids = results[0].boxes.id.cpu().numpy().astype(int)
            classes = results[0].boxes.cls.cpu().numpy().astype(int)
            confidences = results[0].boxes.conf.cpu().numpy()
            
            for box, track_id, cls_id, conf in zip(boxes, track_ids, classes, confidences):
                x1, y1, x2, y2 = map(int, box)
                center_x, center_y = int((x1 + x2) / 2), int((y1 + y2) / 2)
                
                # Logic đếm xe qua đa giác
                is_inside = cv2.pointPolygonTest(roi_points, (center_x, center_y), False)
                if is_inside >= 0 and track_id not in counted_ids:
                    counted_ids.add(track_id)
                    total_vehicles += 1
                    vehicle_counts[cls_id] += 1 
                    roi_color_flash = True      
                
                # Vẽ khung Bounding Box
                color = (0, 255, 0) if cls_id == 4 else (255, 150, 0) 
                cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
                cv2.circle(frame, (center_x, center_y), 5, (0, 0, 255), -1)
                label = f"{class_names[cls_id]} ID:{track_id} {conf:.2f}"
                cv2.putText(frame, label, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

        # Đổi màu đa giác (Nháy sáng vàng khi có xe, bình thường màu đỏ)
        roi_color = (0, 255, 255) if roi_color_flash else (0, 0, 255)
        cv2.polylines(frame, [roi_points], isClosed=True, color=roi_color, thickness=2 if not roi_color_flash else 5)

        # Hiển thị Bảng thống kê
        cv2.rectangle(frame, (10, 10), (450, 220), (0, 0, 0), -1)
        cv2.putText(frame, f"TONG SO XE: {total_vehicles}", (20, 50), cv2.FONT_HERSHEY_DUPLEX, 1.2, (0, 255, 255), 2)
        cv2.putText(frame, f"- Xe May: {vehicle_counts[1]}", (20, 90), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)
        cv2.putText(frame, f"- O To: {vehicle_counts[0]}", (20, 120), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)
        cv2.putText(frame, f"- Xe Tai/Bus: {vehicle_counts[2] + vehicle_counts[3]}", (20, 150), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)
        cv2.putText(frame, f"- XE BA GAC: {vehicle_counts[4]}", (20, 190), cv2.FONT_HERSHEY_DUPLEX, 0.9, (0, 255, 0), 2)

        out.write(frame)

    cap.release()
    out.release()
    print(f"\n✅ ĐÃ XỬ LÝ XONG! Mời bạn mở thư mục KQTEST để xem kết quả:\n{video_output}")